# Simple MNIST Training

In [ ]:
import os

import pandas as pd
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import MNIST
from torchvision.transforms import ToTensor
from tqdm import tqdm

top_dir = "/Users/sauron/MLProjects"
if os.getcwd() != top_dir:
    print(f"Changing working directory to: {top_dir}")
    os.chdir("/Users/sauron/MLProjects")

from common.paths import get_data_dir

In [ ]:
device = (
    torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
)
device

# Parameters

In [ ]:
# Dataset Split
dataset_generator = torch.Generator().manual_seed(42)
train_validation_split = 0.8
batch_size = 5_000

# Optimizer
learning_rate = 1e-3

# Training Loop
epochs = 200

# Get Training Data

In [ ]:
data_dir = get_data_dir()
train_input_dir = data_dir / "MNIST" / "train"
test_input_dir = data_dir / "MNIST" / "test"

data = MNIST(root=train_input_dir, train=True, download=True, transform=ToTensor())
data

# Make a Network

In [ ]:
class SimpleNetwork(nn.Module):
    def __init__(self, num_features: int = 10, num_internal: int = 128) -> None:
        super().__init__()
        self.pipeline = nn.Sequential(
            nn.Flatten(),
            nn.LazyLinear(num_internal),
            nn.ReLU(),
            nn.LazyLinear(num_features),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.pipeline(x)


network = SimpleNetwork().to(device)
network

# Simple Training Loop

In [ ]:
# Dataset Split
train_size = int(train_validation_split * len(data))
validation_size = len(data) - train_size
train_data, validation_data = random_split(
    data,
    [train_size, validation_size],
    generator=dataset_generator,
)

In [ ]:
optimizer = AdamW(network.parameters(), lr=learning_rate)
optimizer

In [ ]:
loss_func = nn.CrossEntropyLoss()
loss_func

In [ ]:
train_dataloader = DataLoader(train_data, batch_size=batch_size, shuffle=True)
validation_dataloader = DataLoader(validation_data, batch_size=batch_size)

In [ ]:
train_result = pd.DataFrame(
    columns=["train_loss", "validation_loss"],
    index=pd.Series(range(epochs), name="epoch"),
)

progress_bar = tqdm(range(epochs))
latest_train_loss = -1.0
latest_validation_loss = -1.0

for epoch in progress_bar:
    progress_bar.set_description(
        f"E: {epoch} | Train: {latest_train_loss:.4f} | Val: {latest_validation_loss:.4f}"
    )

    network.train()
    total_train_loss = 0
    for batch_X, batch_y in train_dataloader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        output = network(batch_X)
        loss = loss_func(output, batch_y)
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
    latest_train_loss = total_train_loss / len(train_dataloader)
    progress_bar.set_description(
        f"E: {epoch} | Train: {latest_train_loss:.4f} | Val: {latest_validation_loss:.4f}"
    )

    network.eval()
    with torch.inference_mode():
        validation_loss = 0
        for batch_X, batch_y in validation_dataloader:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            output = network(batch_X)
            validation_loss += loss_func(output, batch_y)
    latest_validation_loss = validation_loss.item()
    progress_bar.set_description(
        f"E: {epoch} | Train: {latest_train_loss:.4f} | Val: {latest_validation_loss:.4f}"
    )

    train_result.loc[epoch, "train_loss"] = latest_train_loss
    train_result.loc[epoch, "validation_loss"] = latest_validation_loss

In [ ]:
train_result.plot(
    y=["train_loss", "validation_loss"], title="Loss Curves", logy=True, grid=True
)

# Performance Metrics

# Precision, Recall, F1, ROC, etc